# Spalanie jednostek obliczeniowych Colaba dla beki (edycja "AGI")

Problem: sklasyfikować 7 liczb jako "atak" albo "nie atak".
`RandomForestClassifier` robi to w [ml/train.py](../ml/train.py) w
**mniej niż sekundę na CPU** z wynikiem precision=0.973/recall=1.000.

Zamiast tego: **transformer z ~1.6 miliarda parametrów i mechanizmem
uwagi**, trenowany na tych samych 7 liczbach, na najmocniejszym GPU,
jakie Colab da. To nie jest AGI. To jest młot kowalski na komara,
tyle że młot waży tyle co GPT-2-XL i kosztuje realne jednostki
obliczeniowe. Traktuj nazewnictwo w komórkach poniżej jako żart, nie
jako twierdzenie inżynierskie.

**Wymaga runtime z GPU** (najlepiej A100, `Runtime -> Change runtime
type -> A100 GPU`) -- na T4 model prawdopodobnie się nie zmieści w
pamięci.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "BRAK -- wlacz GPU w Runtime, bo to psuje cala zabawe")
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else "-")

In [ ]:
import getpass

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token (scope: repo): ")
REPO = "pamsmediatech-lang/ai-waf-spec"
!git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git /content/ai-waf-spec 2>&1 | tail -5
%cd /content/ai-waf-spec
import sys
sys.path.insert(0, "/content/ai-waf-spec")

In [ ]:
# Ten sam zbior, co dla porzadnego RandomForest -- 7 cech, ~1200 probek.
# 'AGI' ponizej zobaczy dokladnie tyle samo danych co las losowy, ktory
# rozwiazuje to zadanie w mniej niz sekunde. To jest cala roznica miedzy
# inzynierem a kims, kto ma 200 jednostek obliczeniowych do przepalenia.
import numpy as np
import torch

from ml.dataset import FEATURE_NAMES, build_dataset

X_dicts, y = build_dataset(n_benign=600, n_malicious=600, seed=42)
X = np.array([[row[name] for name in FEATURE_NAMES] for row in X_dicts], dtype=np.float32)
y = np.array(y, dtype=np.int64)
print(X.shape, y.shape, "-- tak, to jest cale nasze 'big data'")

In [ ]:
# "AGI" (cudzyslow robi cala robote): traktujemy 7 cech jako sekwencje
# 7 "tokenow", kazdy embedowany do D_MODEL wymiarow, przepuszczany przez
# stos transformer-encoderow z multi-head attention, potem pooling i
# glowa klasyfikujaca. Attention pomiedzy siedmioma skalarami -- bo
# czemu nie.
import torch.nn as nn

D_MODEL = 2048
N_HEADS = 32
N_LAYERS = 32
FFN_DIM = 8192


class TotallyNotAGI(nn.Module):
    def __init__(self, n_features: int, n_classes: int = 2):
        super().__init__()
        # kazda cecha to jeden "token"; embedding to po prostu Linear(1 -> D_MODEL)
        self.token_embed = nn.Linear(1, D_MODEL)
        self.pos_embed = nn.Parameter(torch.randn(n_features, D_MODEL) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=FFN_DIM,
            batch_first=True, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=N_LAYERS)
        self.head = nn.Sequential(nn.LayerNorm(D_MODEL), nn.Linear(D_MODEL, n_classes))

    def forward(self, x):
        tokens = self.token_embed(x.unsqueeze(-1)) + self.pos_embed  # (B, n_features, D_MODEL)
        encoded = self.encoder(tokens)
        pooled = encoded.mean(dim=1)
        return self.head(pooled)


model = TotallyNotAGI(n_features=len(FEATURE_NAMES)).cuda()
n_params = sum(p.numel() for p in model.parameters())
print(f"parametrow: {n_params:,} (~{n_params/1e9:.2f} miliarda) -- do sklasyfikowania 7 liczb")
print(f"to jest {n_params / 1200:,.0f} parametrow na kazda probke treningowa")
print(f"szacowana pamiec (wagi+grad+adam, fp32): ~{n_params * 16 / 1e9:.1f} GB")
print("AGI-o-metr: 0% (to nadal jest klasyfikator siedmiu liczb, tylko drozszy)")

In [ ]:
# Batch size 1 (celowo -- GPU ma sie nudzic miedzy krokami), mixed
# precision zeby w ogole zmiescic sie w VRAM przy tej liczbie
# parametrow, i skromne 15 epok -- przy 1.6 mld parametrow i batch=1,
# 500 epok jak w poprzedniej (mniejszej) wersji zajeloby to reelnie
# wiele godzin. To i tak absurdalnie duzo compute jak na 7 liczb.
import time

X_t = torch.tensor(X).cuda()
y_t = torch.tensor(y).cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler()

EPOCHS = 15
start = time.time()
for epoch in range(EPOCHS):
    perm = torch.randperm(len(X_t))
    total_loss = 0.0
    for i in perm:  # batch size 1, na zlosc
        optimizer.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            out = model(X_t[i:i+1])
            loss = loss_fn(out, y_t[i:i+1])
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    elapsed = time.time() - start
    print(f"epoka {epoch+1:3d}/{EPOCHS} | strata {total_loss/len(X_t):.4f} | "
          f"minelo {elapsed:7.1f}s | 200 jednostek obliczeniowych placze gdzies w tle")
print(f"\nCALOSC: {time.time()-start:.1f}s, {n_params:,} parametrow (~{n_params/1e9:.2f} mld), "
      f"zeby (moze) dogonic RandomForest z ml/train.py, ktory zrobil to samo w <1s na CPU.")

## Rachunek strat

- `ml/train.py` (RandomForest, CPU): **< 1 sekunda**, precision 0.973, recall 1.000
- ten notebook (transformer ~1.6 mld parametrów, A100, attention między 7 liczbami): **od kilkunastu minut do godzin**, zależnie od tego, jak bardzo Colab pozwoli się rozpędzić, wynik najpewniej podobny albo gorszy (1.6 mld parametrów na 1200 próbkach to podręcznikowy przepis na przeuczenie)

AGI nie osiągnięte. Jednostki obliczeniowe: tak. 🎷🔥